# AI-Assisted Momentum Reversal Risk Monitor

## PM Read — 2026-05-29

> **Crowding risk is building in a concentrated long-side theme, but stress has not spread across the book and there is no evidence of forced deleveraging. Review the exposed names; broad de-risking is premature.**

| PM decision field | Current read |
| --- | --- |
| **Severity** | Elevated but contained; focused PM review warranted |
| **Mechanism** | Crowded-theme unwind partially supported; recovery-driven crash weak |
| **Risk Distribution** | Concentrated long-side theme exposure; short-leg pressure remains normal|
| **Why not act broadly** | Selling is still being absorbed; stress has not spread beyond the cluster |
| **Next checks** | Review name-level concentration, selling breadth, liquidity, and independent positioning evidence |


## 1. The PM problem

Momentum weakness is ambiguous. The same drawdown can be ordinary noise, a **recovery-driven reversal**, or a **crowded-position unwind**.

> **Central question:** Is current momentum weakness ordinary noise, a recovery-driven crash setup, or a crowded unwind — and what should the PM monitor next?

This notebook is a product walkthrough. It does **not** predict crash timing or issue a trade instruction.

**Default monitored book:** S&P 500 12-1 long-10 / short-10 (research stand-in).  
**Comparison context only:** Ken French UMD / Daniel–Moskowitz market state.


<details>
<summary><strong>Mechanism primer — optional</strong></summary>

## 2. Two momentum-crash mechanisms

### Mechanism 1 — Recovery-driven momentum crash (Daniel–Moskowitz)

**PM question:** Is the market recovering from a severe drawdown in a way that could produce a sharp loser-stock rebound and damage momentum?

Watch for prior market drawdown, recovery state, loser- or short-leg rebound, beta asymmetry, short-leg losses, and momentum-portfolio drawdown.

### Mechanism 2 — Crowded-position unwind (Khandani–Lo)

**PM question:** Is a crowded momentum trade being reduced or unwound in a way that could amplify losses across similar portfolios?

Watch for crowded or concentrated exposure, unusual reductions in technology or momentum exposure, correlated selling, and possible deleveraging or liquidity pressure.

Do **not** claim forced deleveraging unless the evidence supports it.

| Mechanism | Support | Weaken |
| --- | --- | --- |
| Recovery-driven crash | Panic / severe drawdown → rapid recovery → loser rebound / short-leg pain | Soft recovery without panic; no short-basket stress |
| Crowded unwind | Crowding / concentration + synchronized selling + weak absorption | Selling without crowding; healthy absorption |

</details>


<details>
<summary><strong>Decision workflow — optional</strong></summary>

## 3. How the decision workflow works

```text
Rules-based risk checks → likely mechanism → evidence challenge → PM next checks
```

The PM uses the combined read to choose among:

1. maintain monitoring;
2. inspect the short leg or concentrated exposures;
3. challenge the signal with additional evidence;
4. discuss whether risk escalation is warranted.

**AI evidence layer:** organizes supporting, contradicting, and missing evidence. It does **not** set the risk signal or change the portfolio rules.

</details>


<details>
<summary><strong>Run settings — quant / research</strong></summary>

### Setup

Change `CONFIG` in the next parameter cell, then run all cells. The default recomputed assessment and the primary frozen case now share the **2026-05-29** assessment date for consistency. The primary case still uses a separately reviewed frozen evidence pack; March 2020, January 2024, and cross-case sections are also frozen reference packs and do not change with `CONFIG`.

**LLM interpretation — two runtime modes:**

| Mode | Settings | Behavior |
| --- | --- | --- |
| **Offline deterministic** (default) | `CONFIG.use_llm=False` | No API call. Scorecards, triggers, and FINRA proxy table are unchanged; Evidence Card and PM narratives use calibrated deterministic text. |
| **Live DeepSeek-assisted** | `CONFIG.use_llm=True` + `DEEPSEEK_API_KEY` in `.env` | Injects `DeepSeekEvidenceInterpreter` and `DeepSeekPMResponseInterpreter` into `run_mvp`. FINRA/CFTC proxies enter the Evidence Card model context; PM response is phrased from the bounded category allow-list. Missing key, HTTP failure, or schema validation failure falls back to deterministic text. |

Supported live dates must be trading dates covered by the bundled processed data (currently through 2026-05-29).

</details>


In [1]:
# Bootstrap imports whether the kernel starts in the repository root or notebooks/.
from pathlib import Path
import json
import sys

from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "mvp" / "pipeline.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from " + str(start))

ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.mvp.config import MVPConfig
from src.mvp.pipeline import run_mvp

print("Repository root: .")


Repository root: .


In [2]:
# PM sandbox parameters: edit this block, then Run All.
# use_llm=False (default) = offline deterministic Evidence Card + PM narrative.
# use_llm=True + DEEPSEEK_API_KEY in .env = live DeepSeek for both Evidence Card
# and PM response (injected in the live assessment cell below).
CONFIG = MVPConfig(
    as_of_date="2026-05-29",
    compare_to_date="2026-04-30",
    threshold_profile="default",
    horizon_days=20,
    use_llm=True,  # Live DeepSeek Evidence Card + PM response
)

CASE_PACKS = {
    "current_semi": ROOT / "outputs" / "current_semi_unwind",
    "march_2020": ROOT / "outputs" / "march_2020_reference",
    "quiet_2024": ROOT / "outputs" / "quiet_control_2024",
    "cross_case": ROOT / "outputs" / "cross_case_comparison.md",
}

missing = [
    label
    for label, path in [
        ("current_semi/pm_case_read.md", CASE_PACKS["current_semi"] / "pm_case_read.md"),
        ("current_semi/mechanism_comparison.md", CASE_PACKS["current_semi"] / "mechanism_comparison.md"),
        ("march_2020/pm_case_read.md", CASE_PACKS["march_2020"] / "pm_case_read.md"),
        ("quiet_2024/pm_case_read.md", CASE_PACKS["quiet_2024"] / "pm_case_read.md"),
        ("cross_case_comparison.md", CASE_PACKS["cross_case"]),
    ]
    if not path.exists()
]
if missing:
    raise FileNotFoundError("Missing product packs: " + ", ".join(missing))
print("Live CONFIG:", {
    "as_of_date": CONFIG.as_of_date,
    "compare_to_date": CONFIG.compare_to_date,
    "threshold_profile": CONFIG.threshold_profile,
    "horizon_days": CONFIG.horizon_days,
    "use_llm": CONFIG.use_llm,
})
print("Frozen product packs ready.")


Live CONFIG: {'as_of_date': '2026-05-29', 'compare_to_date': '2026-04-30', 'threshold_profile': 'default', 'horizon_days': 20, 'use_llm': True}
Frozen product packs ready.


## 4. Primary correlated-cluster case

**Primary demo.** Frozen assessment date **2026-05-29** (historical partial read — not a live August assessment).

Use the same reading structure for every case:

1. **Current read** — what is happening?
2. **Mechanism assessment** — which momentum-crash mechanism is supported?
3. **Portfolio implication** — where is the risk located?
4. **Evidence view** — what supports or contradicts the interpretation?
5. **PM interpretation** — what should the PM monitor next?

Expected credible conclusion:

> Localized crowding and meaningful structural pressure are supported; a broad recovery-driven crash or forced unwind is **not** yet confirmed.


In [3]:
display(Markdown((CASE_PACKS["current_semi"] / "pm_case_read.md").read_text()))


# PM case read

**Assessment date:** 2026-05-29  
**Evidence cutoff:** 2026-05-29 16:00 America/New_York  
**Risk horizon:** 20 trading days

**Monitoring severity:** Potential momentum tail risk; elevated but contained

**Staleness warning:** This is a historical partial read and must not be presented as a current 2026-08-03 assessment.  
**Interpretation:** `deterministic-evidence-interpretation-v2` · **PM response:** `deterministic-pm-response-v1`

**Evidence-layer note:** The `CSU-*` records are a separate, manually curated
and human-reviewed cutoff-valid case pack. Active exact-date
classification-cache replay is unavailable for 2026-05-29, so these records
challenge the deterministic snapshot without becoming pipeline-generated
evidence or changing any quantitative field.

## Current read

The core risk indicators are quiet, but concentration and trading patterns warrant focused review. The clearest concern is a partially supported crowded-theme unwind; a broad recovery-driven crash and fundamental repricing remain weak or unconfirmed. Broad action is premature while selling remains contained within the cluster and continues to be absorbed.

## Mechanism comparison

| Lens | Read |
| --- | --- |
| Daniel–Moskowitz recovery crash | **weak** — recovery text (`CSU-2026-015`) without panic, loser-leg rebound, or short-loss confirmation |
| Khandani–Lo crowded unwind | **partially supported** — concentration and turnover are elevated, but selling is still being absorbed and has not spread beyond the cluster |
| Fundamental or sector-specific repricing | **weak** — capex/FCF context without completed reprice; operating results contradict broad deterioration (`CSU-2026-008`) |

## Structured and text alignment

- **Agree:** Localized long-side crowding / correlated-theme stress is the clearest book channel; recovery backdrop exists in text while DM structural completion does not.
- **Conflict / incomplete:** Strong operating results (`CSU-2026-008`) cut against a broad negative fundamental thesis; Prime Book and capex items are human-confirmed but only contextual, so they cannot enter supporting/contradicting ID fields.
- **Layer split:** Quant scorecard = inactive triggers; structural = `crowded_theme_unwind` triggered; trading footprint = potential momentum tail risk, with elevated turnover but no liquidity failure or broad factor spread.

## Where the risk sits

- **Long-side crowding:** only triggered structural mechanism is `crowded_theme_unwind`.
- **Concentration:** portfolio concentration triggered; detected cluster remains the focal long-side channel.
- **Trading footprint:** turnover is elevated, but selling is still being absorbed.
- **Broader strategy drawdown / short basket:** monitored drawdown and short-loss signals are not triggered.

## What is supported

- Structured crowded-theme unwind and concentration stress in the PM book.
- Elevated turnover and concentrated pressure, with no sign that market liquidity is failing.
- Market-recovery component in retrieved text (`CSU-2026-015`).
- Contradicting operating strength at a cluster name (`CSU-2026-008`).

## What remains unconfirmed

- Forced deleveraging, financing pressure, or dealer-inventory stress.
- Factor propagation beyond the detected cluster.
- Liquidity-absorption failure.
- Complete DM sequence (panic + loser-leg rebound + short-leg loss).
- Completed fundamental valuation or earnings reprice.
- Stance-confirmed citation of contextual items `CSU-2026-013`, `CSU-2026-004`, `CSU-2026-005` (MVP citation limitation).

## What would confirm propagation

1. Factor footprint and aligned turnover broaden beyond the active structural channel.
2. Liquidity-absorption failure appears while losses remain synchronized.
3. Independent positioning evidence moves from contextual/mixed toward continued long reduction rather than re-entry.

## What would invalidate the current interpretation

1. Active structural mechanisms return to `not_confirmed` and mechanical state normalizes.
2. Liquidity absorption remains healthy while breadth stays confined.
3. Supplied contradicting evidence materially weakens the crowded-theme monitoring read.

## Why broad action may still be premature

A crowded-theme signal warrants focused review, but broad automatic de-risking is still premature: selling is being absorbed, stress has not spread beyond the cluster, the recovery-crash and fundamental lenses remain weak, and positioning evidence is not strong enough to confirm the narrative. Any assessment of the later semiconductor selloff requires a refreshed portfolio snapshot and evidence from the same cutoff.


### Mechanism lenses (frozen May 29)

Keep Daniel–Moskowitz, Khandani–Lo, and fundamental / sector repricing separate. Do not force a single winner.


In [4]:
display(Markdown((CASE_PACKS["current_semi"] / "mechanism_comparison.md").read_text()))


# Mechanism comparison

**Frozen assessment:** 2026-05-29, 16:00 America/New_York  
**Current market date:** 2026-08-03  
**Scope:** Historical partial evidence pack. It is not a current August market assessment.  
**Interpretation layer:** `deterministic-evidence-interpretation-v2` with compact structural and mechanical context (`evidence-interpretation-prompt-v2`).

**Evidence-layer note:** `CSU-*` denotes the separately curated,
human-reviewed, cutoff-valid case pack. The active exact-date
classification-cache replay is unavailable on 2026-05-29; curated evidence is
interpretive context only and does not alter deterministic fields.

| Lens | Status | Structured support | Text support | Missing evidence |
| --- | --- | --- | --- | --- |
| DM recovery crash | **weak** | Recovery from trough is present, but prior severe-drawdown and high-volatility conditions are false. `bear_market_recovery_crash` is `watch`; `short_book_reversal_crash` is `not_confirmed`; loser outperformance and short-rise breadth are below thresholds. Quantitative recovery and short-loss signals are not triggered. | RBC describes a broad recovery from the March selloff (`CSU-2026-015`, supporting). | No direct prior-loser rebound evidence; no broad panic confirmation; no short-leg loss trigger. |
| Khandani–Lo crowded unwind | **partially supported** | `crowded_theme_unwind` is triggered; portfolio concentration is triggered; the `CIEN`–`COHR`–`LITE` cluster shows broad losses and abnormal-volume confirmation. The trading footprint indicates **potential momentum tail risk** with elevated aligned turnover. Stress has not spread across the factor, selling is still being absorbed, breadth remains healthy, and synchronous winner liquidation is not triggered. | Early-May Prime Book reporting describes an unusually large hedge-fund reduction in technology exposure, led by long sales (`CSU-2026-013`, contextual — discussed, not stance-citable as support). | No direct leverage, financing, dealer-inventory, or forced-liquidation data; no verified factor propagation outside the detected cluster; no liquidity failure. |
| Fundamental or sector-specific repricing | **weak** | The repository fundamental anchor is unavailable, so there is no structured support for a negative fundamental reprice. | Capex and free-cash-flow pressure are visible (`CSU-2026-004`, `CSU-2026-005`, contextual). Strong contemporaneous operating results contradict a broad deterioration thesis (`CSU-2026-008`, contradicting). | No timestamp-valid analyst estimate revisions, valuation de-rating series, or completed cash-flow reprice. |

## Cross-lens conclusion

As of May 29, the strongest supported interpretation remains **localized crowding with potential momentum tail risk**, not a completed DM crash or a demonstrated sector-wide fundamental collapse. Quantitative scorecard triggers are inactive, while concentration and trading-footprint channels are active — those layers must be read separately. Because the later semiconductor selloff began after the cutoff, this pack cannot adjudicate its cause without a refreshed structured date.

## Explicit non-findings

- The statistical correlated cluster is not a validated semiconductor theme.
- Public correlation, turnover, and Prime Book reporting do not establish forced liquidation.
- Strong trailing earnings do not rule out later valuation compression.
- Contextual evidence cannot be formally cited in supporting/contradicting ID fields (MVP limitation).
- No probability, causality claim, or trade recommendation is produced.


## 5. AI evidence view

The AI / evidence layer answers:

- Why might this signal be occurring?
- Which mechanism does the evidence support?
- What evidence argues against the risk interpretation?
- What information should the PM check next?
- How confident should the PM be in the narrative?

Below: supporting / contradicting / not-yet-confirmed items from the frozen semi pack, plus the live quiet-control interpreter path for transparency.


In [5]:
import re

case_read = (CASE_PACKS["current_semi"] / "pm_case_read.md").read_text()

def section(title: str, text: str) -> str:
    pattern = rf"## {re.escape(title)}\n(.*?)(?=\n## |\Z)"
    match = re.search(pattern, text, flags=re.S)
    return match.group(0).strip() if match else f"_Section not found: {title}_"

for title in [
    "What is supported",
    "What remains unconfirmed",
    "Why broad action may still be premature",
]:
    display(Markdown(section(title, case_read)))


## What is supported

- Structured crowded-theme unwind and concentration stress in the PM book.
- Elevated turnover and concentrated pressure, with no sign that market liquidity is failing.
- Market-recovery component in retrieved text (`CSU-2026-015`).
- Contradicting operating strength at a cluster name (`CSU-2026-008`).

## What remains unconfirmed

- Forced deleveraging, financing pressure, or dealer-inventory stress.
- Factor propagation beyond the detected cluster.
- Liquidity-absorption failure.
- Complete DM sequence (panic + loser-leg rebound + short-leg loss).
- Completed fundamental valuation or earnings reprice.
- Stance-confirmed citation of contextual items `CSU-2026-013`, `CSU-2026-004`, `CSU-2026-005` (MVP citation limitation).

## Why broad action may still be premature

A crowded-theme signal warrants focused review, but broad automatic de-risking is still premature: selling is being absorbed, stress has not spread beyond the cluster, the recovery-crash and fundamental lenses remain weak, and positioning evidence is not strong enough to confirm the narrative. Any assessment of the later semiconductor selloff requires a refreshed portfolio snapshot and evidence from the same cutoff.

### Parameterized live assessment — `CONFIG` through `run_mvp`

This section reruns the rules-based assessment for **2026-05-29** by default. Changing `CONFIG` changes this section only; it does not rewrite the reviewed case packs below.

**Positioning hierarchy (how to talk about it):**

1. **Book structure** — concentration / crowded-theme rows (deterministic)
2. **Market trading footprint** — turnover, breadth, and whether selling is still being absorbed (rules-based)
3. **Public positioning proxies** — FINRA short-activity z on the loser basket (public proxy; contextual only)
4. **Reported flows** — Prime Book and similar evidence, used only when the direction is clear

FINRA/CFTC-style proxies provide context only and never change the risk signal. They may strengthen or weaken a **hypothesis** such as short-side crowding, but they cannot establish hedge-fund covering or forced deleveraging.


In [6]:
from src.mvp.crowding_context import build_positioning_snapshot
from src.mvp.deepseek_evidence_interpreter import DeepSeekEvidenceInterpreter
from src.mvp.deepseek_pm_response_interpreter import DeepSeekPMResponseInterpreter
from src.mvp.evidence_interpretation import public_positioning_proxy_items
from src.evidence.deepseek_explainer import _load_dotenv_if_present

if CONFIG.use_llm:
    _load_dotenv_if_present()

evidence_interpreter = DeepSeekEvidenceInterpreter() if CONFIG.use_llm else None
pm_interpreter = DeepSeekPMResponseInterpreter() if CONFIG.use_llm else None
result = run_mvp(
    CONFIG,
    interpreter=evidence_interpreter,
    pm_interpreter=pm_interpreter,
)
evidence = result.deterministic_input
unwind = result.unwind.to_dict()
mechanical = result.mechanical_unwind
interpretation = result.interpretation
pm = result.pm_response

# Same typed proxies the interpretation path receives (presentation + calibration).
positioning = build_positioning_snapshot(
    as_of_date=CONFIG.as_of_date,
    context_elevated=bool(evidence.triggered_quant_signals),
    processed_dir=CONFIG.processed_dir,
)
positioning_proxies = public_positioning_proxy_items(positioning)

scenarios = {row["scenario"]: row["status"] for row in unwind["mechanism_scenarios"]}
tail_risk_state = {
    "FRAGILITY_BUILDING": "potential momentum tail risk",
}.get(mechanical.unwind_state, mechanical.unwind_state.replace("_", " ").lower())

if not positioning_proxies:
    proxy_lines = (
        "_No FINRA public positioning proxy available for this as-of "
        "(panel missing or unavailable)._"
    )
else:
    proxy_rows = []
    for item in positioning_proxies:
        value = item.get("value")
        value_text = f"{value:+.2f}" if isinstance(value, (int, float)) else "—"
        proxy_rows.append(
            f"| `{item['source']}` | `{item['metric']}` | {value_text} | "
            f"`{item.get('state') or '—'}` | {item['reporting_lag']} |"
        )
    proxy_lines = "\n".join(
        [
            "| Source | Metric | Value | State | Reporting lag |",
            "| --- | --- | --- | --- | --- |",
            *proxy_rows,
            "",
            f"**Scope:** {positioning_proxies[0]['relevant_asset_or_portfolio_scope']}",
            "",
            "Contextual only; does not change the scorecard or confirm an unwind.",
            "",
            "**PM read example (if elevated):** public short-activity proxies raise "
            "the relevance of short-side crowding as a *hypothesis*; they do **not** "
            "identify investors or establish active covering / forced deleveraging.",
        ]
    )

interpretation_heading = (
    "Live LLM interpretation — DeepSeek"
    if interpretation and interpretation.use_llm
    else "Deterministic fallback interpretation"
)
pm_heading = (
    "Live LLM PM response — DeepSeek"
    if pm and pm.use_llm
    else "Deterministic fallback PM response"
)
interpretation_text = interpretation.pm_interpretation if interpretation else "unavailable"
pm_state_text = pm.current_state if pm else "unavailable"
pm_why_text = pm.why_not_act_yet if pm else "unavailable"

summary = f"""
**Live parameterized assessment ({evidence.as_of_date})**

| Layer | Read |
| --- | --- |
| UMD / market context | `{evidence.overall_risk_state}` (comparison only) |
| Scorecard triggers | {len(evidence.triggered_quant_signals)} |
| Recovery crash | `{scenarios.get("bear_market_recovery_crash")}` |
| Short-book reversal | `{scenarios.get("short_book_reversal_crash")}` |
| Crowded theme unwind | `{scenarios.get("crowded_theme_unwind")}` |
| Momentum tail-risk state | `{tail_risk_state}` |
| Classification | `{unwind.get("scenario_classification")}` |

**Public positioning proxies** (context only; not in scorecard)

{proxy_lines}

### {interpretation_heading}

> “{interpretation_text}”

*Mode: {"live DeepSeek (`" + interpretation.model_or_prompt_version + "`)" if interpretation and interpretation.use_llm else "offline deterministic (`" + (interpretation.model_or_prompt_version if interpretation else "n/a") + "`)"}*

### {pm_heading}

> **Current PM read:** “{pm_state_text}”
>
> **Why not act yet:** “{pm_why_text}”

*Mode: {"live DeepSeek (`" + pm.model_or_prompt_version + "`)" if pm and pm.use_llm else "offline deterministic (`" + (pm.model_or_prompt_version if pm else "n/a") + "`)"}*
"""
display(Markdown(summary))



**Live parameterized assessment (2026-05-29)**

| Layer | Read |
| --- | --- |
| UMD / market context | `normal` (comparison only) |
| Scorecard triggers | 0 |
| Recovery crash | `watch` |
| Short-book reversal | `not_confirmed` |
| Crowded theme unwind | `triggered` |
| Momentum tail-risk state | `potential momentum tail risk` |
| Classification | `crowded_momentum_unwind` |

**Public positioning proxies** (context only; not in scorecard)

| Source | Metric | Value | State | Reporting lag |
| --- | --- | --- | --- | --- |
| `FINRA` | `short_interest_ratio_z` | +2.24 | `elevated` | same trading day as as-of |
| `FINRA` | `short_interest_utilisation_z` | +2.54 | `elevated` | same trading day as as-of |
| `FINRA` | `short_volume_share_z` | -2.39 | `depressed` | same trading day as as-of |

**Scope:** Momentum loser / short-basket public-data proxy universe (not observed book ownership)

Contextual only; does not change the scorecard or confirm an unwind.

**PM read example (if elevated):** public short-activity proxies raise the relevance of short-side crowding as a *hypothesis*; they do **not** identify investors or establish active covering / forced deleveraging.

### Deterministic fallback interpretation

> “Point-in-time evidence is unavailable, so the quantitative state cannot be corroborated or contradicted by the retrieval layer. Lens read: DM recovery crash remains watch on structural channels and not confirmed for short-book reversal; Khandani-Lo crowded unwind is triggered; fundamental repricing stays unconfirmed without a structured fundamental anchor. Momentum tail-risk state is potential momentum tail risk, with liquidity absorption failure absent.”

*Mode: offline deterministic (`deterministic-evidence-interpretation-v2`)*

### Live LLM PM response — DeepSeek

> **Current PM read:** “No deterministic escalation signals are active; the crowded theme unwind mechanism is triggered, but the short book reversal crash is not confirmed, and the overall risk state is normal. Maintain posture and monitor.”
>
> **Why not act yet:** “No broad mechanical unwind is confirmed, and the non-triggered signals do not indicate an active short book reversal. The elevated short interest proxies are contextual but not proof of covering or forced deleveraging, so we maintain current posture and monitor for confirmation.”

*Mode: live DeepSeek (`pm-response-prompt-v5`)*


## 6. 2020 historical validation

Primary historical validation case (**2020-03-24**). Purpose: show that when a known momentum-reversal episode occurred, recovery-crash indicators behaved coherently — not to claim full predictive performance.

Look for the sequence: severe drawdown → rapid recovery → short-leg pain / beta asymmetry → momentum losses. Crowded unwind remains secondary / unconfirmed in this pack.


In [7]:
display(Markdown((CASE_PACKS["march_2020"] / "pm_case_read.md").read_text()))


# PM case read — March 2020 reference

**Assessment date:** 2020-03-24  
**Evidence cutoff:** 2020-03-24 16:00 America/New_York  
**Risk horizon:** 20 trading days

**Interpretation:** `deterministic-evidence-interpretation-v2` · **Prompt:** `evidence-interpretation-prompt-v2` · **PM response:** `deterministic-pm-response-v1`

**Evidence-layer note:** `M20-*` denotes a separate manually curated,
cutoff-valid historical reference pack. Active exact-date
classification-cache replay is unavailable on 2020-03-24; the curated records
support historical interpretation but are not pipeline-retrieved evidence and
cannot change the deterministic state.

## Current read

As of the March 24 freeze, the case is most consistent with a **Daniel–Moskowitz panic-recovery momentum-crash setup**, not a confirmed Khandani–Lo crowded unwind. Quantitative recovery and short-leg loss channels are already triggered, while structural crowded-theme unwind is not confirmed. Broad automatic de-risking still requires PM review: absorption has not failed, and short-book reversal remains only on watch.

## Why the DM lens is stronger or weaker

DM is stronger here than in the May 29, 2026 semi case because structured panic, severe prior drawdown, high volatility, recovery-from-trough, and `bear_market_recovery_crash=triggered` all align, with book stress in `short_loss_in_recovery` and `short_minus_long_beta_gap`. Supporting text is policy/market-stress and recovery-oriented Fed releases (`M20-2020-001`, `M20-2020-005`). The lens is not fully complete: `short_book_reversal_crash` is only **watch**, and the text pack has no direct loser-leg rebound article.

## Where the portfolio risk sits

- **Short basket / loser-leg rebound:** `short_loss_in_recovery` triggered; main PM vulnerability is `short_basket`.
- **Beta gap:** `short_minus_long_beta_gap` triggered.
- **Broader momentum drawdown:** portfolio drawdown worsened but remained not triggered.
- **Structural unwind:** active scenario `bear_market_recovery_crash`; `crowded_theme_unwind` not confirmed.
- **Trading footprint:** potential momentum tail risk; factor stress is elevated, but aligned turnover is not and selling is still being absorbed.

## What is supported

- Panic-elevated market state with met severe-drawdown, high-volatility, and recovery conditions.
- Triggered high-volatility recovery and short-leg loss in the PM book.
- Fed credit-market support actions (`M20-2020-001`).
- Fed pandemic-hardship / recovery-support package near the trough (`M20-2020-005`).
- Contradicting limit on forced-deleveraging overclaim from discount-window borrowing (`M20-2020-004`).

## What remains unconfirmed

- Direct short covering narrative in text evidence.
- Crowded positioning or forced hedge-fund deleveraging.
- Liquidity-absorption failure.
- Factor propagation as a KL episode (`crowded_theme_unwind` not confirmed).
- Completed fundamental valuation reprice of the momentum book.
- Formal citation of contextual items `M20-2020-002/003/006/007` as supporting IDs (known MVP limitation).

## What would confirm the DM mechanism

1. `short_book_reversal_crash` moves from watch to triggered as loser-leg rebound breadth confirms.
2. Short-basket losses remain elevated while the recovery regime persists.
3. Timestamp-valid evidence of prior-loser rebound / short covering joins the policy-recovery backdrop.

## What would invalidate the DM interpretation

1. Recovery stalls and high-volatility-recovery / short-loss triggers reverse without broadening.
2. Stress re-centers on a confirmed crowded-theme liquidation with absorption failure.
3. Book losses prove idiosyncratic and detach from the panic-recovery sequence.

## Comparison with the current semi case

March 2020 is primarily a panic-to-recovery / short-basket reference: DM structural and quant channels are active, while KL remains unconfirmed. By contrast, the May 29, 2026 semi case is more closely associated with long-side crowding and potential momentum tail risk, while selling is still being absorbed and broader factor spread remains unresolved; DM is incomplete.


### Mechanism lenses (March 2020)


In [8]:
display(Markdown((CASE_PACKS["march_2020"] / "mechanism_comparison.md").read_text()))


# Mechanism comparison — March 2020 reference

**Frozen assessment:** 2020-03-24, 16:00 America/New_York  
**Comparison date:** 2020-02-28  
**Interpretation layer:** `deterministic-evidence-interpretation-v2` / prompt `evidence-interpretation-prompt-v2`  
**Scope:** Small historical reference pack. Not a full March 2020 research study.

**Evidence-layer note:** `M20-*` records come from the separately curated,
cutoff-valid historical reference pack. Exact-date classification-cache replay
is unavailable on this date, and the curated text cannot change deterministic
metrics or mechanism states.

| Lens | Structured support | Text support | Missing evidence | Current read |
| --- | --- | --- | --- | --- |
| Daniel–Moskowitz | Market state `panic_elevated`. Severe prior drawdown met (`≈-34%` vs `-20%` gate), high realized volatility met, rapid recovery-from-trough met. `bear_market_recovery_crash` **triggered**. Quant triggers: `high_volatility_recovery`, `short_loss_in_recovery`, `short_minus_long_beta_gap`. `short_book_reversal_crash` is **watch**. Scenario class: `panic_recovery_momentum_crash`. | Fed credit-market support and FOMC-linked actions (`M20-2020-001`); Fed pandemic-hardship package aimed at limiting losses and promoting recovery (`M20-2020-005`). Liquidity/market-functioning facilities are contextual (`M20-2020-002`, `003`, `007`). | No contemporaneous desk report of named loser-leg rebound or short covering in this pack; short-book reversal remains watch rather than triggered. | **supported** (partial on short-book completion) |
| Khandani–Lo | `crowded_theme_unwind` **not_confirmed**. The trading footprint indicates **potential momentum tail risk** through elevated factor stress, but aligned turnover is not elevated and selling is still being absorbed. Portfolio concentration alone does not establish crowded thematic liquidation. | Dollar swaps, PDCF, and discount-window language show funding/liquidity strain (`M20-2020-002`, `003`, `007`), not crowded positioning. Rising discount-window borrowing (`M20-2020-004`) weakens a forced-quant-deleveraging claim. | No direct crowded ownership, synchronized systematic selling, or forced hedge-fund liquidation evidence in the reused official corpus. | **weak / unconfirmed** |
| Fundamental repricing | Repository fundamental anchor unavailable / not confirmatory for a completed valuation reprice. | Pandemic economic disruption and recovery-oriented policy (`M20-2020-005`); Treasury/IRS tax-day delay under COVID emergency (`M20-2020-006`). | No timestamp-valid earnings-revision or sector-reprice series tied to the PM momentum book. | **partial** (macro shock present; completed reprice not shown) |

## Explicit answers

- **Prior broad-market panic?** Yes in structured outputs (`panic_elevated`; severe drawdown and high volatility conditions met).
- **Broad recovery?** Yes: recovery-from-trough condition met; `high_volatility_recovery` triggered; Fed text on 2020-03-23 targets a swift recovery (`M20-2020-005`).
- **Loser-leg rebound risk visible?** Yes in the book via triggered `short_loss_in_recovery` and beta-gap; structural short-book reversal remains **watch**, and the text pack lacks direct loser-leg journalism.
- **Direct crowding or deleveraging evidence?** No. Liquidity facilities ≠ confirmed crowded unwind (`M20-2020-004` limits that claim).
- **Broader than generic pandemic news?** Yes for DM: panic + recovery + short-leg loss triggers are book/mechanism-specific, not only virus headlines.

## Cross-lens conclusion

March 24, 2020 is primarily a **Daniel–Moskowitz panic-recovery reference**: structured DM channels are active, while KL crowded unwind is not confirmed and fundamental evidence remains a pandemic macro overlay rather than a completed security-level reprice. No probability, causality claim, or trade recommendation is produced.


## 7. 2024 quiet control

Control case (**2024-01-05**). Same rules, different conclusion: incomplete recovery preconditions, no confirmed crowded unwind, contained short-leg pressure. Escalation is not justified.

This is how the product shows it is **selective rather than permanently alarmist**.


In [9]:
display(Markdown((CASE_PACKS["quiet_2024"] / "pm_case_read.md").read_text()))


# PM case read — 2024 quiet control

**Assessment date:** 2024-01-05  
**Comparison date:** 2023-12-01  
**Evidence cutoff:** 2024-01-05 16:00 America/New_York  
**Role in product demo:** Quiet control — shows the monitor does not escalate every soft momentum period into a crash setup.  
**Interpretation:** `deterministic-evidence-interpretation-v2` · **PM response:** `deterministic-pm-response-v1`

## Current read

As of 5 January 2024, the monitored momentum book is in a soft-bear / low-vol backdrop (`bear_low_volatility`) with **no scorecard triggers** and **no confirmed crash mechanism**. A partial recovery precondition is visible, but the severe-drawdown and high-volatility gates that complete a Daniel–Moskowitz setup are absent. Crowded-theme unwind is not confirmed. The right PM posture is to maintain monitoring, not escalate.

## Mechanism assessment

| Lens | Read |
| --- | --- |
| Daniel–Moskowitz recovery crash | **watch / incomplete** — recovery-from-trough is met, but prior severe drawdown and high realized volatility are not; `bear_market_recovery_crash` remains watch only |
| Khandani–Lo crowded unwind | **not confirmed** — no active crowded-theme scenario; mechanical state is `NORMAL`; liquidity-absorption failure is absent |
| Short-leg / rebound pressure | **contained** — `short_loss_in_recovery`, `short_minus_long_beta_gap`, and `short_book_reversal_crash` are not triggered |

## Portfolio implication

- **UMD / market backdrop:** soft-bear, low volatility — comparison context only, not a book score.
- **PM scorecard:** 0 of 4 indicators triggered; drawdown (~−9%) remains above the material stress gate.
- **Where risk would appear if conditions worsened:** short basket under a true recovery-crash sequence, or long-side crowding if a theme cluster began to liquidate synchronously with absorption stress.
- **Today:** neither channel is confirmed.

## Evidence view

- **Supports a quiet read:** employment, income, and GDP releases in the exact-date pack are ordinary macro context, not panic-recovery or crowded-unwind confirmation.
- **Contradicts a crash narrative:** no triggered recovery-crash or crowded-unwind mechanism; mechanical footprint is not elevated.
- **Mixed / limited:** one Reuters soft-open item after stronger jobs data (`reuters-2024-01-05-wall-street`) is market color, not mechanism evidence.
- **Missing:** observed positioning, leverage, financing, and live institutional retrieval — same standing limitations as other cases.

## PM interpretation

Maintain the current posture and keep watching for a completed recovery-crash sequence or a confirmed crowded-theme channel. Escalation is not justified on this date.

### What to monitor next

1. Does `bear_market_recovery_crash` move from watch to triggered (severe drawdown + high vol + recovery together)?
2. Do short-leg losses or the beta gap trigger during an early-recovery regime?
3. Does `crowded_theme_unwind` confirm with absorption stress, rather than an isolated structural row?

### Why this differs from 2020

In March 2020 the panic-recovery sequence and short-leg loss channels were active. On 2024-01-05 those conditions are incomplete or absent, so the same rules stay selective rather than permanently alarmist.


### Mechanism lenses (January 2024)


In [10]:
display(Markdown((CASE_PACKS["quiet_2024"] / "mechanism_comparison.md").read_text()))


# Mechanism comparison — 2024 quiet control

**Frozen assessment:** 2024-01-05, 16:00 America/New_York  
**Comparison date:** 2023-12-01  
**Interpretation layer:** `deterministic-evidence-interpretation-v2`  
**Scope:** Default demo negative-control date. Used to show selectivity, not to claim a risk event.

| Lens | Status | Structured support | Text support | Missing evidence |
| --- | --- | --- | --- | --- |
| DM recovery crash | **watch / incomplete** | Recovery-from-trough met (~15%). Prior severe drawdown not met (~−14% vs −20% gate). High realized volatility not met. `bear_market_recovery_crash=watch`. Quant recovery and short-loss signals not triggered. | Macro releases (employment, income, GDP) are ordinary context, not panic-recovery journalism. | No completed panic → recovery → loser-rebound sequence. |
| Khandani–Lo crowded unwind | **not confirmed** | `crowded_theme_unwind=not_confirmed`. Active mechanism list empty. Mechanical state `NORMAL`; factor footprint and aligned turnover not elevated; liquidity-absorption failure false. One structural row (`synchronous_winner_liquidation`) can print without confirming the full crowded-unwind scenario. | No crowding / deleveraging narrative in the exact-date pack. | No ownership, leverage, financing, or forced-liquidation observation. |
| Short-leg pressure | **contained** | `short_loss_in_recovery`, `short_minus_long_beta_gap`, and `short_book_reversal_crash` are not triggered. Portfolio drawdown (~−9%) is not at the material gate. | Soft-open color after stronger jobs data does not establish short-basket stress. | No loser-leg rebound confirmation. |

## Cross-lens conclusion

Classification is `normal_drawdown`. The monitor sees a soft backdrop and an incomplete recovery precondition, then **stops short of escalation**. That selectivity is the point of the quiet-control case.

## Explicit non-findings

- Soft momentum / mild drawdown ≠ crash setup.
- A single elevated structural metric ≠ confirmed crowded unwind.
- UMD `bear_low_volatility` is comparison context only; it is not a PM-book probability.
- No probability, causality claim, or trade recommendation is produced.


## 8. Cross-case comparison

One compact table across the three product cases. Values come from repository outputs, not hard-coded demo claims.


In [11]:
display(Markdown(CASE_PACKS["cross_case"].read_text()))


# Cross-case comparison

Derived from repository case packs and `run_mvp` outputs. Mechanism labels are descriptive reads, not crash probabilities.

| Question | Current semi case (2026-05-29) | 2020 validation (2020-03-24) | 2024 quiet control (2024-01-05) |
| --- | --- | --- | --- |
| Recovery mechanism (Daniel–Moskowitz) | Partial / watch — recovery text without panic, loser rebound, or short-loss confirmation | Strongly present — `bear_market_recovery_crash` triggered; panic, severe drawdown, high vol, recovery aligned | Not present as a completed setup — recovery precondition only; severe drawdown and high vol unmet |
| Crowded unwind evidence (Khandani–Lo) | Contextual / partially supported — `crowded_theme_unwind` triggered; concentration and potential momentum tail risk, while selling is still being absorbed | Secondary / unconfirmed — liquidity facilities ≠ crowded positioning | Limited — crowded-theme scenario not confirmed; trading footprint is normal |
| Short-leg pressure | Contained on scorecard; risk sits more in long-side crowding | Severe — `short_loss_in_recovery` and beta-gap triggered; short-book reversal on watch | Contained — short-loss, beta-gap, and short-book reversal not triggered |
| Evidence confidence | Mixed — localized crowding supported; broad crash and forced unwind unconfirmed | Historically coherent — mechanism indicators line up with a known reversal episode | Low-risk / quiet — ordinary macro context; no confirmed crash channel |
| PM workflow | Monitor and investigate concentrated / theme exposures | Escalate review of recovery-crash and short-basket channels | Maintain monitoring — escalation not justified |

## How to read the table

1. **Current semi** is the primary live-style product demo: localized crowding pressure without a completed recovery crash.
2. **2020** shows that when a historically important momentum reversal occurred, the recovery-crash indicators behaved coherently.
3. **2024** shows the same rules staying quiet when the mechanism is incomplete.

Sources:

- `outputs/current_semi_unwind/pm_case_read.md`
- `outputs/current_semi_unwind/mechanism_comparison.md`
- `outputs/march_2020_reference/pm_case_read.md`
- `outputs/march_2020_reference/mechanism_comparison.md`
- `outputs/quiet_control_2024/pm_case_read.md`
- `outputs/quiet_control_2024/mechanism_comparison.md`
- `outputs/research_validation/episode_fingerprints.md`


## 9. Limitations and next steps

- Default PM book uses survivorship-biased current SPY membership; not a live holdings plug-in yet.
- UMD / DM state is comparison context only — never a PM-book crash probability.
- Crowding and mechanical layers are public-data proxies, not observed ownership, leverage, financing, or dealer inventory.
- **FINRA / CFTC-style public positioning proxies** may enter the interpretation layer as typed contextual evidence. They never enter scorecards, thresholds, mechanism triggers, or escalation rules, and must not be read as investor identity, covering, or forced deleveraging.
- Evidence is exact-date cached replay, not institutional live retrieval.
- Mechanism scenarios are descriptive rules without out-of-sample predictive validation.
- The AI layer organizes and challenges evidence; it cannot change deterministic values or triggers.
- Forced deleveraging is never inferred from correlation or turnover alone.

**Production path** (holdings plug-in, observed crowding, financing / flow overlays): see `docs/production_path.md`.

**Reviewer docs:** `docs/methodology.md` · `docs/limitations.md` · `docs/demo_walkthrough.md`.
